# 📦 AWS Supply Chain Orders — Full Analysis Notebook
**Dataset:** `aws_supply_chain_orders_raw.csv`  
**Columns:** order_id · warehouse · region · product · order_qty · order_date · delivery_date · delivery_time_days · status  
**Rows:** 155 orders across 3 warehouses (WH-A, WH-B, WH-C) and 5 products

---
> Run cells top to bottom. Each section is self-contained with comments explaining every line.


## 1. 📥 Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)        # show all columns
pd.set_option('display.float_format', '{:.2f}'.format)  # 2 decimal places
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print("✅ Imports done")

✅ Imports done


## 2. 📂 Load & Inspect the Dataset

In [2]:
df = pd.read_csv("aws_supply_chain_orders_raw.csv")

print("Shape:", df.shape)           # (rows, columns)
print("Columns:", df.columns.tolist())
df.head(5)                          # first 5 rows

Shape: (155, 9)
Columns: ['order_id', 'warehouse', 'region', 'product', 'order_qty', 'order_date', 'delivery_date', 'delivery_time_days', 'status']


,order_id,warehouse,region,product,order_qty,order_date,delivery_date,delivery_time_days,status
0,ORD-1000,WH-C,West,Router,42,2026-02-16 06:22:56.185879,2026-03-24 06:22:56.186200,4.00,Pending
1,ORD-1001,WH-A,West,Laptop,30,2026-02-25 06:22:56.185905,2026-03-15 06:22:56.186205,12.00,Pending
2,ORD-1002,WH-C,South,Switch,19,2026-03-13 06:22:56.185910,2026-03-13 06:22:56.186207,13.00,Pending
3,ORD-1003,WH-C,North,Server,17,2026-02-14 06:22:56.185913,2026-03-01 06:22:56.186209,9.00,Pending
4,ORD-1004,WH-A,East,Laptop,19,2026-02-18 06:22:56.185916,2026-03-11 06:22:56.186211,3.00,Delivered


In [ ]:
df.info()                             # dtypes + non-null counts + memory

In [ ]:
df.describe()                         # stats for numeric columns only

In [ ]:
# Describe ALL columns including strings
df.describe(include='all')

In [ ]:
# Unique value counts per categorical column
for col in ['warehouse', 'region', 'product', 'status']:
    print(f"{col}: {df[col].unique()}")

In [ ]:
df.sample(5, random_state=42)         # random 5 rows

## 3. 🧹 Null Value Detection & Handling

In [3]:
# Count nulls per column
df.isnull().sum()

order_id               0
warehouse              0
region                 5
product                0
order_qty              0
order_date             0
delivery_date          0
delivery_time_days    11
status                 0
dtype: int64

In [4]:
# Null percentage per column
(df.isnull().sum() / len(df) * 100).round(2).rename("null %")

order_id             0.00
warehouse            0.00
region               3.23
product              0.00
order_qty            0.00
order_date           0.00
delivery_date        0.00
delivery_time_days   7.10
status               0.00
Name: null %, dtype: float64

In [5]:
# See the actual rows with nulls in delivery_time_days
df[df['delivery_time_days'].isnull()]

,order_id,warehouse,region,product,order_qty,order_date,delivery_date,delivery_time_days,status
18,ORD-1018,WH-B,South,Storage,25,2026-01-28 06:22:56.185942,2026-02-25 06:22:56.186237,NaN,Delivered
22,ORD-1022,WH-B,North,Laptop,16,2026-03-17 06:22:56.185949,2026-03-04 06:22:56.186243,NaN,Delivered
56,ORD-1056,WH-C,North,Storage,36,2026-03-18 06:22:56.186017,2026-03-16 06:22:56.186308,NaN,Delayed
58,ORD-1058,WH-C,East,Switch,31,2026-03-13 06:22:56.186021,2026-03-23 06:22:56.186311,NaN,Delivered
82,ORD-1082,WH-B,North,Laptop,12,2026-01-29 06:22:56.186062,2026-03-05 06:22:56.186350,NaN,Delivered
109,ORD-1109,WH-B,West,Router,45,2026-01-23 06:22:56.186108,2026-03-16 06:22:56.186395,NaN,Pending
124,ORD-1124,WH-C,North,Router,36,2026-02-09 06:22:56.186134,2026-02-24 06:22:56.186420,NaN,Delayed
126,ORD-1126,WH-C,West,Switch,49,2026-03-17 06:22:56.186137,2026-03-12 06:22:56.186424,NaN,Delivered
129,ORD-1129,WH-B,West,Storage,24,2026-03-02 06:22:56.186143,2026-03-21 06:22:56.186429,NaN,Delayed
136,ORD-1136,WH-B,East,Server,23,2026-02-17 06:22:56.186155,2026-03-18 06:22:56.186442,NaN,Delivered


In [10]:
# Option A: Fill delivery_time_days nulls with median
df['delivery_time_days'] = df['delivery_time_days'].fillna(df['delivery_time_days'].median())

# Option B: Fill region nulls with 'Unknown'
df['region'] = df['region'].fillna('Unknown')

print("Nulls remaining:", df.isnull().sum().sum())
df.tail()

Nulls remaining: 0


,order_id,warehouse,region,product,order_qty,order_date,delivery_date,delivery_time_days,status
150,ORD-1101,WH-C,South,Router,29,2026-02-02 06:22:56.186094,2026-03-06 06:22:56.186381,3.00,Delivered
151,ORD-1082,WH-B,North,Laptop,12,2026-01-29 06:22:56.186062,2026-03-05 06:22:56.186350,8.50,Delivered
152,ORD-1031,WH-C,South,Server,37,2026-03-05 06:22:56.185974,2026-03-05 06:22:56.186259,14.00,Delivered
153,ORD-1064,WH-B,South,Laptop,45,2026-02-06 06:22:56.186031,2026-03-05 06:22:56.186321,13.00,Delivered
154,ORD-1067,WH-B,West,Storage,24,2026-03-17 06:22:56.186036,2026-02-26 06:22:56.186326,4.00,Delivered


## 4. 🔧 Fix Data Types

In [ ]:
# Convert date strings → datetime objects
df['order_date']    = pd.to_datetime(df['order_date'])
df['delivery_date'] = pd.to_datetime(df['delivery_date'])

# Convert status to categorical (saves memory, enables cat operations)
df['status'] = df['status'].astype('category')

print(df.dtypes)

In [ ]:
# Extract date parts into new columns
df['order_year']    = df['order_date'].dt.year
df['order_month']   = df['order_date'].dt.month
df['order_weekday'] = df['order_date'].dt.day_name()

df[['order_date', 'order_year', 'order_month', 'order_weekday']].head()

## 5. ➕ Creating Derived / Computed Columns

In [ ]:
# Actual delivery days (from raw timestamps)
df['actual_days'] = (df['delivery_date'] - df['order_date']).dt.days

# Delivery gap: positive = late, negative = early
df['delivery_gap'] = df['actual_days'] - df['delivery_time_days']

# Boolean: was the order delayed vs on-time?
df['is_delayed'] = df['status'] == 'Delayed'
df['is_delivered'] = df['status'] == 'Delivered'

# Quantity tier using pd.cut (bin into low/medium/high)
df['qty_tier'] = pd.cut(df['order_qty'],
    bins=[0, 15, 30, 50],
    labels=['low', 'medium', 'high'])

df[['order_id', 'order_qty', 'qty_tier', 'actual_days', 'delivery_gap', 'is_delayed']].head(8)

In [ ]:
# np.where — conditional column (like Excel IF)
df['urgency'] = np.where(df['delivery_time_days'] <= 5, 'urgent', 'standard')

# np.select — multi-condition labeling
conditions = [
    df['delivery_time_days'] <= 5,
    df['delivery_time_days'] <= 10,
    df['delivery_time_days'] > 10
]
choices = ['fast', 'normal', 'slow']
df['speed_label'] = np.select(conditions, choices)

df[['order_id', 'delivery_time_days', 'urgency', 'speed_label']].head(8)

## 6. 🔍 Filtering, Selecting & Slicing

In [ ]:
# Single condition
df[df['status'] == 'Delayed']

In [ ]:
# Multiple conditions with & (AND) and | (OR)
df[(df['status'] == 'Delayed') & (df['region'] == 'North')]

In [ ]:
# .isin() — match multiple values at once
df[df['product'].isin(['Laptop', 'Server'])]

In [ ]:
# ~ means NOT
df[~df['status'].isin(['Delivered'])]

In [ ]:
# .loc — label-based: select rows AND specific columns
df.loc[df['warehouse'] == 'WH-A', ['order_id', 'product', 'order_qty', 'status']]

In [1]:
# .iloc — position-based: rows 10–20, first 4 columns
df.iloc[10:20, 0:4]

NameError: name 'df' is not defined

In [ ]:
# .query() — readable SQL-style filtering
df.query("order_qty > 40 and status == 'Pending'")

## 7. 🔃 Sorting & Ranking

In [ ]:
# Sort by single column
df.sort_values('order_qty', ascending=False).head(10)

In [ ]:
# Multi-column sort: region A-Z, then delivery time descending
df.sort_values(['region', 'delivery_time_days'], ascending=[True, False]).head(10)

In [ ]:
# Rank orders by delivery time
df['delivery_rank'] = df['delivery_time_days'].rank(ascending=True)

# Top 5 fastest deliveries
df.nsmallest(5, 'delivery_time_days')[['order_id', 'product', 'warehouse', 'delivery_time_days']]

In [ ]:
# Top 5 largest orders
df.nlargest(5, 'order_qty')[['order_id', 'product', 'warehouse', 'order_qty']]

## 8. 📊 GroupBy — Supply Chain KPIs

In [ ]:
# Total quantity ordered per warehouse
df.groupby('warehouse')['order_qty'].sum()

In [ ]:
# Average delivery time per region
df.groupby('region')['delivery_time_days'].mean().round(2)

In [ ]:
# Multiple aggregations at once
df.groupby('warehouse').agg(
    total_orders   = ('order_id', 'count'),
    total_qty      = ('order_qty', 'sum'),
    avg_delivery   = ('delivery_time_days', 'mean'),
    max_delivery   = ('delivery_time_days', 'max'),
    delayed_count  = ('is_delayed', 'sum')
).round(2)

In [ ]:
# Delay rate % by warehouse
delay_rate = df.groupby('warehouse')['is_delayed'].mean() * 100
delay_rate.round(1).rename("delay_rate_%")

In [ ]:
# Product performance: total qty + avg delivery
df.groupby('product').agg(
    total_qty      = ('order_qty', 'sum'),
    avg_delivery   = ('delivery_time_days', 'mean'),
    order_count    = ('order_id', 'count')
).sort_values('total_qty', ascending=False).round(2)

In [ ]:
# Multi-level groupby: warehouse × product
df.groupby(['warehouse', 'product'])['order_qty'].sum().unstack(fill_value=0)

## 9. 🗂️ Pivot Tables

In [ ]:
# Units ordered: warehouse (rows) vs product (columns)
pd.pivot_table(df,
    values   = 'order_qty',
    index    = 'warehouse',
    columns  = 'product',
    aggfunc  = 'sum',
    fill_value = 0,
    margins  = True,          # adds row/column totals
    margins_name = 'Total'
)

In [ ]:
# Average delivery time: region vs status
pd.pivot_table(df,
    values  = 'delivery_time_days',
    index   = 'region',
    columns = 'status',
    aggfunc = 'mean'
).round(2)

In [ ]:
# Cross-tab: count of orders per warehouse per status
pd.crosstab(df['warehouse'], df['status'], margins=True)

## 10. 🔤 String / Text Operations (.str accessor)

In [ ]:
# Extract warehouse letter (WH-A → A)
df['wh_code'] = df['warehouse'].str.replace('WH-', '')

# Lowercase
df['product_lower'] = df['product'].str.lower()

# Check if order_id starts with 'ORD'
df['valid_id'] = df['order_id'].str.startswith('ORD')

# Extract order number as integer
df['order_num'] = df['order_id'].str.extract(r'(\d+)').astype(int)

df[['order_id', 'wh_code', 'product_lower', 'valid_id', 'order_num']].head(6)

In [ ]:
# String contains (case-insensitive)
df[df['product'].str.contains('er', case=False)]   # Router, Server, Storage

## 11. 📅 Time Series Analysis

In [ ]:
# Monthly order volume
monthly_orders = df.set_index('order_date').resample('ME')['order_qty'].sum()
monthly_orders.plot(kind='bar', title='Monthly Order Quantity')
plt.xlabel('Month'); plt.ylabel('Total Qty')
plt.tight_layout(); plt.show()

In [ ]:
# Orders per weekday
weekday_counts = df['order_weekday'].value_counts()
order_days = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
weekday_counts.reindex(order_days, fill_value=0).plot(kind='bar', color='steelblue')
plt.title('Orders by Day of Week')
plt.tight_layout(); plt.show()

In [ ]:
# Rolling 7-day average of order qty
df_ts = df.set_index('order_date').sort_index()
df_ts['order_qty'].rolling('7D').mean().plot(label='7-day rolling avg')
df_ts['order_qty'].plot(alpha=0.3, label='raw')
plt.title('Order Quantity Over Time'); plt.legend()
plt.tight_layout(); plt.show()

## 12. 🔢 NumPy Operations

In [ ]:
arr = df['order_qty'].values        # pandas Series → numpy array

print("Mean:     ", np.mean(arr).round(2))
print("Median:   ", np.median(arr))
print("Std Dev:  ", np.std(arr).round(2))
print("Variance: ", np.var(arr).round(2))
print("Min/Max:  ", np.min(arr), "/", np.max(arr))
print("Quartiles:", np.percentile(arr, [25, 50, 75]))
print("Sum:      ", np.sum(arr))

In [ ]:
# Z-score normalization of delivery time
dt = df['delivery_time_days'].values
z_scores = (dt - np.mean(dt)) / np.std(dt)
df['delivery_z'] = z_scores

# Outliers: z > 2 or z < -2
outliers = df[np.abs(df['delivery_z']) > 2]
print(f"Outlier orders: {len(outliers)}")
outliers[['order_id', 'delivery_time_days', 'delivery_z']]

In [ ]:
# Log transform (useful for skewed data)
df['qty_log'] = np.log1p(df['order_qty'])

# Clip values to a range (remove extreme outliers)
df['qty_clipped'] = np.clip(df['order_qty'], 5, 45)

# Correlation between order qty and delivery time
qty = df['order_qty'].values
dtime = df['delivery_time_days'].values
corr_matrix = np.corrcoef(qty, dtime)
print(f"Correlation (qty vs delivery_time): {corr_matrix[0,1]:.3f}")

In [ ]:
# Boolean mask with NumPy
fast_mask = df['delivery_time_days'].values <= 5
print("Fast deliveries (≤5 days):", np.sum(fast_mask))
print("Any order > 48 qty?", np.any(df['order_qty'].values > 48))
print("All qty positive?", np.all(df['order_qty'].values > 0))

## 13. 📈 Visualizations

In [ ]:
# Bar chart: total qty by warehouse
wh_qty = df.groupby('warehouse')['order_qty'].sum().sort_values(ascending=False)
ax = wh_qty.plot(kind='bar', color=['#2196F3','#FF9800','#4CAF50'])
ax.set_title('Total Order Quantity by Warehouse')
ax.set_xlabel('Warehouse'); ax.set_ylabel('Total Qty')
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x()+0.1, p.get_height()+5), fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
# Status distribution: pie chart
status_counts = df['status'].value_counts()
colors = ['#4CAF50','#FF9800','#F44336']
status_counts.plot(kind='pie', autopct='%1.1f%%', colors=colors, startangle=90)
plt.title('Order Status Distribution'); plt.ylabel('')
plt.tight_layout(); plt.show()

In [ ]:
# Histogram: delivery time distribution
df['delivery_time_days'].plot(kind='hist', bins=14, edgecolor='white', color='#5C6BC0')
plt.axvline(df['delivery_time_days'].mean(), color='red', linestyle='--', label='mean')
plt.axvline(df['delivery_time_days'].median(), color='orange', linestyle='--', label='median')
plt.legend(); plt.title('Delivery Time Distribution')
plt.xlabel('Days'); plt.tight_layout(); plt.show()

In [ ]:
# Heatmap: avg delivery time per warehouse × product
pivot = df.pivot_table(values='delivery_time_days', index='warehouse',
                       columns='product', aggfunc='mean')
fig, ax = plt.subplots(figsize=(10, 3))
im = ax.imshow(pivot.values, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)));  ax.set_yticklabels(pivot.index)
plt.colorbar(im, ax=ax, label='Avg Delivery Days')
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j, i, f"{pivot.values[i,j]:.1f}", ha='center', va='center', fontsize=10)
ax.set_title('Avg Delivery Time (days) — Warehouse × Product')
plt.tight_layout(); plt.show()

In [ ]:
# Scatter: order qty vs delivery time (colored by status)
colors_map = {'Pending': '#FF9800', 'Delivered': '#4CAF50', 'Delayed': '#F44336'}
for status, group in df.groupby('status', observed=True):
    plt.scatter(group['order_qty'], group['delivery_time_days'],
                label=status, color=colors_map[status], alpha=0.7, s=50)
plt.xlabel('Order Quantity'); plt.ylabel('Delivery Time (days)')
plt.title('Order Qty vs Delivery Time by Status')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Box plot: delivery time by warehouse
df.boxplot(column='delivery_time_days', by='warehouse',
           patch_artist=True, figsize=(8,4))
plt.suptitle('')
plt.title('Delivery Time Distribution by Warehouse')
plt.xlabel('Warehouse'); plt.ylabel('Days')
plt.tight_layout(); plt.show()

## 14. 🔗 Merge, Join & Concat

In [ ]:
# Simulate a second orders file and concatenate
df_copy = df.sample(20, random_state=7).copy()
df_copy['order_id'] = df_copy['order_id'].str.replace('ORD-', 'ORD-Q2-')
combined = pd.concat([df, df_copy], ignore_index=True)
print("Combined shape:", combined.shape)

In [ ]:
# Simulate a warehouse capacity lookup table and merge
warehouse_info = pd.DataFrame({
    'warehouse' : ['WH-A', 'WH-B', 'WH-C'],
    'capacity'  : [500, 300, 450],
    'city'      : ['Mumbai', 'Delhi', 'Bangalore']
})
df_merged = pd.merge(df, warehouse_info, on='warehouse', how='left')
df_merged[['order_id','warehouse','city','capacity','order_qty']].head(5)

In [ ]:
# Left join vs inner join
df_inner = pd.merge(df, warehouse_info, on='warehouse', how='inner')
df_left  = pd.merge(df, warehouse_info, on='warehouse', how='left')
print("Inner:", df_inner.shape, "  Left:", df_left.shape)

## 15. ⚡ apply(), map(), lambda

In [ ]:
# map: recode one column's values
region_map = {'North': 'N', 'South': 'S', 'East': 'E', 'West': 'W', 'Unknown': 'UNK'}
df['region_code'] = df['region'].map(region_map)
df[['region', 'region_code']].drop_duplicates()

In [ ]:
# apply on a single column with lambda
df['qty_doubled'] = df['order_qty'].apply(lambda x: x * 2)

# apply on multiple columns (axis=1 = row-wise)
df['delivery_status_flag'] = df.apply(
    lambda row: 'CRITICAL' if row['is_delayed'] and row['order_qty'] > 35 else 'OK',
    axis=1
)
df[['order_id', 'order_qty', 'is_delayed', 'delivery_status_flag']].head(8)

In [ ]:
# applymap (now called map in pandas 2.x) for element-wise on whole df
numeric_df = df[['order_qty', 'delivery_time_days']].copy()
numeric_df.map(lambda x: round(x, 0))

## 16. 🗑️ Duplicates, Index & Reset

In [ ]:
# Check for duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# Check for duplicate order_ids
print("Duplicate order_ids:", df['order_id'].duplicated().sum())

# Drop duplicates
df_clean = df.drop_duplicates()
df_clean = df.drop_duplicates(subset=['order_id'])

In [ ]:
# Set a meaningful index
df_indexed = df.set_index('order_id')
df_indexed.loc['ORD-1010']           # access row by order ID

# Reset back to default int index
df_reset = df_indexed.reset_index()

## 17. 💾 Export Results

In [2]:
# Save cleaned dataframe to CSV
df.to_csv("supply_chain_cleaned.csv", index=False)
print("Saved: supply_chain_cleaned.csv")

NameError: name 'df' is not defined

In [ ]:
# Save KPI summary to Excel
summary = df.groupby('warehouse').agg(
    total_orders  = ('order_id', 'count'),
    total_qty     = ('order_qty', 'sum'),
    avg_delivery  = ('delivery_time_days', 'mean'),
    delay_rate_pct= ('is_delayed', lambda x: round(x.mean()*100, 1))
).reset_index()

summary.to_excel("warehouse_kpi_summary.xlsx", index=False)
print("Saved: warehouse_kpi_summary.xlsx")
summary

In [ ]:
# Multiple sheets in one Excel file
with pd.ExcelWriter("supply_chain_report.xlsx", engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='Raw Data', index=False)
    summary.to_excel(writer, sheet_name='Warehouse KPIs', index=False)
    df.groupby('product').agg(
        total_qty=('order_qty','sum'),
        avg_delivery=('delivery_time_days','mean')
    ).reset_index().to_excel(writer, sheet_name='Product Analysis', index=False)

print("Saved: supply_chain_report.xlsx with 3 sheets")

## 18. 🏁 Final Supply Chain KPI Dashboard

In [ ]:
total_orders    = len(df)
total_qty       = df['order_qty'].sum()
delay_rate      = (df['status'] == 'Delayed').mean() * 100
avg_delivery    = df['delivery_time_days'].mean()
worst_warehouse = df.groupby('warehouse')['delivery_time_days'].mean().idxmax()
top_product     = df.groupby('product')['order_qty'].sum().idxmax()

print("=" * 45)
print("    AWS SUPPLY CHAIN — KPI SUMMARY")
print("=" * 45)
print(f"  Total Orders        : {total_orders}")
print(f"  Total Units Ordered : {total_qty:,}")
print(f"  Delay Rate          : {delay_rate:.1f}%")
print(f"  Avg Delivery Time   : {avg_delivery:.1f} days")
print(f"  Slowest Warehouse   : {worst_warehouse}")
print(f"  Top Product (qty)   : {top_product}")
print("=" * 45)